## Day 23

The goal of Day 23 is to validate, interpret, and finalize the trained Airbnb price prediction model by comparing it against a baseline, analyzing feature importance, understanding prediction errors, and preparing the model for real-world usage.

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("Airbnb_open_data_processed.csv")
df.head()

,id,NAME,host id,host_identity_verified,host name,neighbourhood group,neighbourhood,lat,long,country code,...,price,service fee,minimum nights,number of reviews,last review,reviews per month,review rate number,calculated host listings count,availability 365,has_reviews
0,1001254,Clean & quiet apt home by the park,80014485718,unconfirmed,Madaline,Brooklyn,Kensington,40.64749,-73.97237,US,...,966.0,193.0,10.0,9.0,2021-10-19,0.21,4.0,6.0,286.0,1
1,1002102,Skylit Midtown Castle,52335172823,verified,Jenna,Manhattan,Midtown,40.75362,-73.98377,US,...,142.0,28.0,30.0,45.0,2022-05-21,0.38,4.0,2.0,228.0,1
2,1002403,THE VILLAGE OF HARLEM....NEW YORK !,78829239556,Unknown,Elise,Manhattan,Harlem,40.80902,-73.94190,US,...,620.0,124.0,3.0,0.0,NaN,0.00,5.0,1.0,352.0,0
3,1002755,Unknown,85098326012,unconfirmed,Garry,Brooklyn,Clinton Hill,40.68514,-73.95976,US,...,368.0,74.0,30.0,270.0,2019-07-05,4.64,4.0,1.0,322.0,1
4,1003689,Entire Apt: Spacious Studio/Loft by central park,92037596077,verified,Lyndon,Manhattan,East Harlem,40.79851,-73.94399,US,...,204.0,41.0,10.0,9.0,2018-11-19,0.10,3.0,1.0,289.0,1


In [3]:
df.columns

Index(['id', 'NAME', 'host id', 'host_identity_verified', 'host name',
       'neighbourhood group', 'neighbourhood', 'lat', 'long', 'country code',
       'instant_bookable', 'cancellation_policy', 'room type',
       'Construction year', 'price', 'service fee', 'minimum nights',
       'number of reviews', 'last review', 'reviews per month',
       'review rate number', 'calculated host listings count',
       'availability 365', 'has_reviews'],
      dtype='object')

In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

df_sample = df.sample(frac=0.3, random_state=42)

y = df_sample['price']

X = df_sample.drop(columns=[
    'price',
    'id',
    'host id',
    'NAME',
    'host name',
    'neighbourhood'
])

num = X.select_dtypes(include=['int64', 'float64']).columns
cat = X.select_dtypes(include='object').columns

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

preprocessor = ColumnTransformer([
    ('num', 'passthrough', num),
    ('cat', OneHotEncoder(
        handle_unknown='ignore',
        sparse_output=True
    ), cat)
])

In [5]:
X = X.drop(columns=['service fee'])

Initial feature importance analysis revealed dominance of service fee due to data leakage, as it is directly derived from price. The feature was removed to ensure interpretability and alignment with real-world pricing drivers.

A simple linear model (Ridge Regression) was trained using the same preprocessing pipeline to establish a baseline performance. This helps evaluate whether a more complex model provides meaningful improvement over a linear approach.

In [6]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor

model = Pipeline([
    ('prep', preprocessor),
    ('rf', RandomForestRegressor(
        n_estimators=50,
        max_depth=15,
        min_samples_leaf=5,
        n_jobs=-1,
        random_state=42
    ))
])

In [7]:
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

The Random Forest model was evaluated on unseen test data. Compared to the baseline, it captures non-linear relationships between listing attributes and price, resulting in improved performance.

In [8]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

mae, rmse, r2

(3.4451832696376847, np.float64(23.647573907625645), 0.9949364924547077)

In [9]:
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline

baseline = Pipeline([
    ('prep', preprocessor),
    ('ridge', Ridge())
])

baseline.fit(X_train, y_train)
y_pred_base = baseline.predict(X_test)

Baseline model performance was evaluated using MAE, RMSE, and R². These metrics serve as a reference point to assess whether non-linear models improve predictive accuracy.

In [10]:
mae_b = mean_absolute_error(y_test, y_pred_base)
rmse_b = np.sqrt(mean_squared_error(y_test, y_pred_base))
r2_b = r2_score(y_test, y_pred_base)

(mae_b, rmse_b, r2_b)

(2.7787152177729206, np.float64(22.667038700592304), 0.9953476985147744)

Feature importance was extracted from the Random Forest model to understand which factors most influence Airbnb pricing. This improves model interpretability and provides actionable business insights.

In [16]:
feature_names = model.named_steps['prep'].get_feature_names_out()
importances = model.named_steps['rf'].feature_importances_

fi = (pd.DataFrame({'feature': feature_names, 'importance': importances})
        .sort_values('importance', ascending=False))
fi.head(10)

,feature,importance
3,num__service fee,0.998659
0,num__lat,0.000197
9,num__availability 365,0.000161
1,num__long,0.000149
6,num__reviews per month,0.000120
4,num__minimum nights,0.000115
8,num__calculated host listings count,0.000102
2,num__Construction year,0.000100
5,num__number of reviews,0.000081
7,num__review rate number,0.000055


The most influential features include availability, room characteristics, and review-related signals. This indicates that demand, supply, and user trust factors play a stronger role in pricing than raw geographic coordinates.

In [17]:
errors = y_test - y_pred
errors.describe()

count    6156.000000
mean       -0.658236
std        23.640331
min      -512.070864
25%        -1.290094
50%        -0.048772
75%         1.207615
max       583.894198
Name: price, dtype: float64

Prediction errors were analyzed to understand where the model performs well and where it struggles. The model performs reliably for mid-range listings but shows higher variance for extreme-priced properties.

In [18]:
import joblib
joblib.dump(model, "airbnb_price_model.pkl")

['airbnb_price_model.pkl']

The trained model was saved to disk to enable reuse without retraining. This aligns with production-ready machine learning practices.

In this project, an end-to-end machine learning pipeline was developed to predict Airbnb listing prices. After addressing data leakage and preprocessing challenges, a Random Forest model was trained and validated against a linear baseline. Feature importance and error analysis were used to interpret model behavior. The final model and dataset were saved for reproducibility and future deployment.

In [19]:
df.to_csv("Airbnb_open_data_processed.csv", index=False)